____
# INTERPOLATE AND ROTATE SWOT DATA ON COLOC POINTS

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import dask.dataframe as dd
import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_2km, add_mask_inside_swot, add_grid_metrics, build_swath_polygon
#from diagnosis import drifters_sources

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

In [6]:
if True:
    from dask.distributed import Client
    from dask_jobqueue import PBSCluster
    from dask import config

    config.set({"distributed.comm.timeouts.connect": "200s"})
    cluster = PBSCluster(cores=28, processes=28, walltime="01:00:00")
    # cluster = PBSCluster(cores=20, processes=20, walltime='02:00:00')#8
    w = cluster.scale(jobs=1)
else:
    from dask.distributed import Client, LocalCluster

    cluster = LocalCluster()

client = Client(cluster)
client

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: http://10.148.0.74:8787/status,
Dashboard: http://10.148.0.74:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.148.0.74:51775,Workers: 0
Dashboard: http://10.148.0.74:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [5]:
cluster.close()

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/

_______________
# CHOOSE coloc sources

In [11]:
drifters_sources = 'all_med_variational_10min_v0.nc'

# Drifters param
#dt = '10d'
dt = '12h'
drifter_preprocess = 'spectral_decomp' # '', 'spectral_decomp', 'low_pass'

if drifter_preprocess == True : drifters_sources = 'spectral_decomp_'+ drifters_sources

colocs_source = f'{dt}_{drifter_preprocess}_{drifters_sources}'.replace('.nc', '.csv')

# Drifters
ddf = dd.read_csv(os.path.join(zarr_dir,'coloc_files', 'drifters', f'drifterscoloc_'+colocs_source), dtype={'drifter_id':str}, parse_dates=['datetime']).set_index('row_number').repartition(npartitions=56)

ddf = ddf[(~((ddf.pass_number==3)& (ddf.cycle_number==568))) & (~((ddf.pass_number==16)&((ddf.cycle_number ==508)|(ddf.cycle_number ==513)|(ddf.cycle_number ==534)|(ddf.cycle_number ==554)|(ddf.cycle_number ==568))))].persist()

df = ddf.compute()

In [4]:
ddf =ddf.persist()

_________
# Functions

In [8]:
ggd_variables = ['cvl_mean_dynamic_topography_cnes_cls_22',
                 'cvl_mean_sea_surface_cnes_22_hybrid',
                 'cvl_ocean_tide_fes_2022',
                 'cvl_ssha_reference',
                 'duacs_ssha_karin_2_calibrated',
                 'duacs_ssha_karin_2_filtered',]


variables =[#'ancillary_surface_classification_flag',
    'cross_track_distance',
    'distance_to_coast',
    'duacs_editing_flag',
    #'duacs_phase_screen',
    #'duacs_phase_screen_orbit',
    #'duacs_phase_screen_static',
    'duacs_relative_vorticity',
    'duacs_speed_meridional',
    'duacs_speed_meridional_abs',
    'duacs_speed_zonal',
    'duacs_speed_zonal_abs',
    'duacs_strain',
    'duacs_xcal',
    'sig0_karin_2',
    'phi',
    'swh_model', 
    'ssh_karin_uncert',
    'pass_number',
]

# For swot 2km
from swot import interp_dss, rotate
def interp_coloc_one_cycle(dfr_, dsalti):#, vars_to_rotate=[]):
    cycle = dfr_.cycle_number.values.mean()
    print(cycle)
    
    if  'cycle_number' in dsalti : 
        dsalti_ = dsalti.sel(cycle_number = cycle)
    else:
        dsalti_ = dsalti
        
    # ATTENTION : NEED TO REMOVE VARIABLES FOR WHICH LONGITUDE, LATITUDE ARE NOT COORDS
    dropv = []
    if 'pass_number' in dsalti_.keys() : dropv +=['pass_number']
    if 'cycle_number' in dsalti_.keys() : dropv +=['cycle_number']
    if 'npts' in dsalti_.keys() : dropv +=['npts']
    if 'cutoff' in dsalti_.keys() : dropv +=['cutoff']
    
    df_interp = interp_dss(dsalti_.drop_vars(dropv), dfr_.longitude.values, dfr_.latitude.values)
    df_out = pd.concat([dfr_.reset_index()[['row_number', 'longitude']].set_index('longitude'), df_interp.set_index('longitude')], axis=1).reset_index().set_index('row_number')
    #for v in vars_to_rotate :
    #    df_out[v[O]], df_out[v[1]] = rotate(df_out[v[O]], df_out[v[1]], df_out(phi))
    return df_out
    
def coloc_swot2km_cycle(dfr, dsalti):
    meta = interp_coloc_one_cycle(dfr[dfr.cycle_number==531].compute(), dsalti)
    
    df_out = dfr.groupby('cycle_number').apply(interp_coloc_one_cycle, dsalti, meta=meta)
    #DF = []
    #for cycle in dfr[dfr.pass_number==swath].cycle_number.unique():
    #    try : 
    #        dfr_ = dfr.where((dfr.pass_number==swath)&(dfr.cycle_number==cycle)).dropna()
    #        dsalti_ = dsalti.sel(cycle_number=cycle)
    #        DF.append(interp_coloc_one_cycle(dfr_, dsalti_, cycle))
    #    except : 
    #        print('no', cycle)
    #        continue
        #print(cycle)
    #df_out = pd.concat(DF)
    #print(dsalti.cutoff.values)
    if 'npts' in dsalti.keys(): df_out['npts'] = dsalti.npts.values
    if 'cutoff' in dsalti.keys(): df_out['cutoff'] = int(dsalti.cutoff.values)
    return df_out

# For L4
def coloc_L4(dfr, dsalti):
    lon = df.longitude.values
    lat = df.latitude.values
    t = pd.to_datetime(df.datetime).values
    ds0 = dsalti.interp(longitude=('z', lon), latitude=('z',lat), time= ('z',t))
    df0 = ds0.to_dataframe()
    df0.index.names = ['row_number'] 
    return df0

    

___________
# Find all L4 files

In [6]:
alti_files = glob(os.path.join(zarr_dir, 'before_coloc', 'L4_sealevel', '*'))
alti_files

['/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/L4_sealevel/L4_noswot_regional.nc',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/L4_sealevel/L4_withnadirswot.nc',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/L4_sealevel/L4_noswot_global.nc']

In [7]:
for f in alti_files : 
    dsalti = xr.open_dataset(f)
    dfout = coloc_L4(df, dsalti)
    path = os.path.join(zarr_dir, "coloc_files",'alti', 'alticoloc_'+f.split('/')[-1].replace('.nc', '_'+colocs_source))
    dfout.to_csv(path)
    print(path)

/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_L4_noswot_regional_10d_spectral_decomp_all_med_variational_10min_v0.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_L4_withnadirswot_10d_spectral_decomp_all_med_variational_10min_v0.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_L4_noswot_global_10d_spectral_decomp_all_med_variational_10min_v0.csv


___________
# Find all preprocessed swot-2km files

In [9]:
alti_files = glob(os.path.join(zarr_dir, 'before_coloc', 'preprocessed_swot', 'swot2km', '*', '*'))
alti_files = [f.replace('pass3', 'pass*') for f in alti_files if 'pass3' in f]
alti_files = [f for f in alti_files if ('general' in f)|('diff_only' in f)]
alti_files

['/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot2km/general/pass*_swot2km_general.nc',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot2km/diff_only/pass*_swot2km_diff_only.nc']

# Interpolate with the data of the nearest cycle


In [12]:
for i in range(len(alti_files)) :

    path = alti_files[i].replace('/'.join(alti_files[i].split('/')[6:10]), 'coloc_files/alti').replace('pass*', 'alticoloc').replace('.nc', '_'+colocs_source)
    #if os.path.isfile(path) :
    #   print(f'file already exists : {path}')
    #   continue
        
    # Alti files
    files = glob(alti_files[i])
    D=[]# for over pass_number
    for f in files:
        try : 
            dsalti = xr.open_dataset(f).compute()
            if 'phi' in dsalti.variables: dsalti = dsalti.reset_coords(['phi', 'dx', 'dy'])
            pass_number = dsalti.pass_number.values
            print(pass_number)
            ddf_ = ddf[ddf.pass_number==pass_number].persist()
            df_out = coloc_swot2km_cycle(ddf_, dsalti).compute().reset_index().set_index('row_number').sort_index()
            df_out['pass_number']=int(pass_number)
            D.append(df_out)
        except :
            assert False, f'pb with {f}'
    pd.concat(D, axis=0).to_csv(path)
    print(path)
    

16
531.0


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 347.92 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


3
531.0


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 370.56 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_swot2km_general_12h_spectral_decomp_all_med_variational_10min_v0.csv
3
531.0


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 589.96 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


16
531.0


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 553.87 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_swot2km_diff_only_12h_spectral_decomp_all_med_variational_10min_v0.csv


In [9]:

d = xr.open_dataset('/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot2km/xarray_diff/pass16_swot2km_xarray_diff.nc')

In [14]:
df[df.pass_number==16].cycle_number.unique()

array([502, 503, 504, 505, 506, 507, 509, 510, 511, 512, 514, 515, 516,
       517, 518, 519, 520, 521, 522, 523, 524, 525, 529, 530, 531, 532,
       533, 535, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546,
       547, 548, 549, 550, 551, 552, 553, 555, 556, 557, 558, 559, 560,
       561, 562, 563, 564, 565, 566, 567, 569, 570, 571, 572, 573, 574,
       575, 576, 577])

______________________
# Find L3-250m

In [10]:
alti_dir = glob(os.path.join(zarr_dir, 'before_coloc', 'preprocessed_swot', 'swot250m', '*', '*'))
#alti_files = [f.replace('pass3', 'pass*') for f in alti_files if 'pass3' in f]
alti_dir = np.unique(['/'.join(f.split('/')[:-1]) for f in alti_dir])
alti_dir = [f for f in alti_dir if ('general' in f)|('diff_only' in f)]
alti_dir

['/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/diff_only',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/general']

In [11]:
def wrapper_interp_one_cycle(df, dir_):
    #print(list(df.keys()))
    swath = int(df.reset_index().pass_number.mean())
    cycle = int(df.reset_index().cycle_number.mean())
    path = os.path.join(dir_, f'{int(swath)}_{int(cycle)}.nc')
    try : 
        dsalti = xr.open_dataset(path)
        if 'phi' in dsalti : dsalti = dsalti.reset_coords(['phi'])
        print(path)
    except : 
        assert False, path
    dfout = interp_coloc_one_cycle(df, dsalti)
    return dfout
    
dir_ = alti_dir[-1]
dfr = df.where((df.pass_number==3)&(df.cycle_number==531)).dropna().iloc[0:100]
meta = dfr.groupby(['pass_number', 'cycle_number']).apply(wrapper_interp_one_cycle, dir_)
meta

/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/general/3_531.nc
531.0


KeyError: "'cycle_number' is not a valid dimension or coordinate for Dataset with dimensions FrozenMappingWarningOnValuesAccess({'num_lines': 4161, 'num_pixels': 519})"

In [ ]:
for dir_ in alti_dir :
    path = dir_.replace('before_coloc/preprocessed_swot', 'coloc_files/alti').replace('swot250m/', 'alticoloc_swot250m_')+'_'+colocs_source
    #if os.path.isfile(path) : continue
    
    dfr = df.where((df.pass_number==3)&(df.cycle_number==500)).dropna().iloc[0:100]
    meta = dfr.groupby(['pass_number', 'cycle_number'], observed=True).apply(wrapper_interp_one_cycle, dir_)
    
    df_out = ddf.groupby(['pass_number', 'cycle_number'], observed=True).apply(wrapper_interp_one_cycle, dir_,  meta = meta).reset_index().compute()
    path = dir_.replace('before_coloc/preprocessed_swot', 'coloc_files/alti').replace('swot250m/', 'alticoloc_swot250m_')+'_'+colocs_source
    df_out.to_csv(path)
    print(path)
    

In [20]:
path = '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_swot250m_general_12h_spectral_decomp_all_med_variational_10min_v0.csv'
df = pd.read_csv(path)
df.columns

Index(['Unnamed: 0', 'pass_number', 'cycle_number', 'row_number', 'longitude',
       'latitude', 'ancillary_surface_classification_flag',
       'cross_track_distance', 'distance_to_coast', 'duacs_editing_flag',
       'duacs_phase_screen', 'duacs_phase_screen_orbit',
       'duacs_phase_screen_static', 'duacs_relative_vorticity',
       'duacs_speed_meridional', 'duacs_speed_meridional_abs',
       'duacs_speed_zonal', 'duacs_speed_zonal_abs', 'duacs_strain',
       'duacs_xcal', 'sig0_karin_2', 'phi', 'swh_model'],
      dtype='object')

In [12]:
dfr.pass_number

row_number
53513    3.0
53514    3.0
53515    3.0
53516    3.0
53517    3.0
        ... 
53608    3.0
53609    3.0
53610    3.0
53611    3.0
53612    3.0
Name: pass_number, Length: 100, dtype: float64